# DO upvote guys

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import gc
import warnings
import os
warnings.filterwarnings('ignore')

print('Libraries imported')

Libraries imported


In [2]:
TRAIN_PATH = '/kaggle/input/ts-forecasting/train.parquet'
TEST_PATH = '/kaggle/input/ts-forecasting/test.parquet'

if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = 'train.parquet'
    TEST_PATH = 'test.parquet'

VAL_THRESHOLD = 3500
FORECAST_WINDOWS = [1, 3, 10, 25]

# SAME HYPERPARAMETERS AS 0.2408 MODEL (proven to work)
LGB_PARAMS = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.015,
    'n_estimators': 4200,
    'num_leaves': 80,
    'min_child_samples': 200,
    'feature_fraction': 0.6,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'lambda_l1': 0.1,
    'lambda_l2': 10.0,
    'verbosity': -1
}

print('Configuration complete - using proven hyperparameters')

Configuration complete - using proven hyperparameters


## Same Features as 0.2408 Model

In [3]:
def add_lag_features(df, value_cols=['feature_al', 'feature_am', 'feature_cg', 'feature_by'], lags=[1, 3, 5, 10, 25]):
    df = df.sort_values(['code', 'sub_code', 'sub_category', 'horizon', 'ts_index'])
    for col in value_cols:
        for lag in lags:
            df[f'{col}_lag_{lag}'] = df.groupby(['code', 'sub_code', 'sub_category', 'horizon'])[col].shift(lag)
    return df

def add_rolling_features(df, value_cols=['feature_al', 'feature_am'], windows=[5, 10, 20]):
    df = df.sort_values(['code', 'sub_code', 'sub_category', 'horizon', 'ts_index'])
    for col in value_cols:
        for window in windows:
            df[f'{col}_roll_mean_{window}'] = df.groupby(['code', 'sub_code', 'sub_category', 'horizon'])[col].transform(lambda x: x.rolling(window, min_periods=1).mean())
            df[f'{col}_roll_std_{window}'] = df.groupby(['code', 'sub_code', 'sub_category', 'horizon'])[col].transform(lambda x: x.rolling(window, min_periods=1).std())
    return df

def add_trend_features(df, value_cols=['feature_al', 'feature_am'], windows=[10, 20]):
    df = df.sort_values(['code', 'sub_code', 'sub_category', 'horizon', 'ts_index'])
    def rolling_slope(series, window):
        def calc_slope(y):
            if len(y) < 2:
                return 0
            x = np.arange(len(y))
            return np.polyfit(x, y, 1)[0] if len(y) > 1 else 0
        return series.rolling(window, min_periods=2).apply(calc_slope, raw=True)
    for col in value_cols:
        for window in windows:
            df[f'{col}_trend_{window}'] = df.groupby(['code', 'sub_code', 'sub_category', 'horizon'])[col].transform(lambda x: rolling_slope(x, window))
    return df

def build_enhanced_features(data, enc_stats=None):
    df = data.copy()
    if enc_stats is not None:
        for c in ['sub_category', 'sub_code']:
            df[c + '_enc'] = df[c].map(enc_stats[c]).fillna(enc_stats['global_mean'])
    df['d_al_am'] = df['feature_al'] - df['feature_am']
    df['r_al_am'] = df['feature_al'] / (df['feature_am'] + 1e-7)
    df['d_cg_by'] = df['feature_cg'] - df['feature_by']
    norm_cols = ['feature_al', 'feature_am', 'feature_cg', 'feature_by', 'd_al_am']
    for col in norm_cols:
        g = df.groupby('ts_index')[col]
        df[col + '_cs'] = (df[col] - g.transform('mean')) / (g.transform('std') + 1e-7)
    df['t_cycle'] = np.sin(2 * np.pi * df['ts_index'] / 100)
    df = add_lag_features(df)
    df = add_rolling_features(df)
    df = add_trend_features(df)
    for col in ['feature_al', 'feature_am']:
        df[f'{col}_diff_1'] = df.groupby(['code', 'sub_code', 'sub_category', 'horizon'])[col].diff(1)
        df[f'{col}_rank'] = df.groupby('ts_index')[col].rank(pct=True)
    df = df.fillna(0)
    return df

def get_feature_columns(df):
    exclude_cols = {'id', 'code', 'sub_code', 'sub_category', 'horizon', 'ts_index', 'weight', 'y_target'}
    return [c for c in df.columns if c not in exclude_cols]

print('Feature engineering ready - same as 0.2408 model')

Feature engineering ready - same as 0.2408 model


In [4]:
def weighted_rmse_score(y_target, y_pred, w):
    y_target = np.array(y_target)
    y_pred = np.array(y_pred)
    w = np.array(w)
    denom = np.sum(w * (y_target ** 2))
    if denom <= 0:
        return 0.0
    numerator = np.sum(w * ((y_target - y_pred) ** 2))
    ratio = numerator / denom
    return float(np.sqrt(1.0 - np.clip(ratio, 0.0, 1.0)))

print('Metric ready')

Metric ready


In [5]:
print('Computing statistics...')
temp = pd.read_parquet(TRAIN_PATH, columns=['sub_category', 'sub_code', 'y_target', 'ts_index'])
train_only = temp[temp.ts_index <= VAL_THRESHOLD]
train_stats = {
    'sub_category': train_only.groupby('sub_category')['y_target'].mean().to_dict(),
    'sub_code': train_only.groupby('sub_code')['y_target'].mean().to_dict(),
    'global_mean': train_only['y_target'].mean()
}
del temp, train_only
gc.collect()
print('Statistics computed')

Computing statistics...
Statistics computed


## KEY CHANGE: 5-Seed Ensemble (was 2)

In [6]:
def train_single_horizon(horizon):
    print(f'Training Horizon {horizon}')
    tr_df = pd.read_parquet(TRAIN_PATH).query(f'horizon == {horizon}')
    te_df = pd.read_parquet(TEST_PATH).query(f'horizon == {horizon}')
    tr_df = build_enhanced_features(tr_df, train_stats)
    te_df = build_enhanced_features(te_df, train_stats)
    feature_cols = get_feature_columns(tr_df)
    print(f'Using {len(feature_cols)} features')
    fit_mask = tr_df.ts_index <= VAL_THRESHOLD
    val_mask = tr_df.ts_index > VAL_THRESHOLD
    X_fit = tr_df.loc[fit_mask, feature_cols]
    y_fit = tr_df.loc[fit_mask, 'y_target']
    w_fit = tr_df.loc[fit_mask, 'weight']
    X_hold = tr_df.loc[val_mask, feature_cols]
    y_hold = tr_df.loc[val_mask, 'y_target']
    w_hold = tr_df.loc[val_mask, 'weight']
    val_pred = np.zeros(len(y_hold))
    tst_pred = np.zeros(len(te_df))
    
    # 5 SEEDS instead of 2 - main improvement
    seeds = [42, 2024, 12345, 99, 420]
    for seed in seeds:
        mdl = lgb.LGBMRegressor(**LGB_PARAMS, random_state=seed)
        mdl.fit(X_fit, y_fit, sample_weight=w_fit, eval_set=[(X_hold, y_hold)], eval_sample_weight=[w_hold], callbacks=[lgb.early_stopping(200, verbose=False)])
        val_pred += mdl.predict(X_hold) / len(seeds)
        tst_pred += mdl.predict(te_df[feature_cols]) / len(seeds)
    
    val_score = weighted_rmse_score(y_hold, val_pred, w_hold)
    print(f'Horizon {horizon} Score: {val_score:.5f}')
    test_ids = te_df['id'].values
    del tr_df, te_df, X_fit, y_fit, X_hold, y_hold, mdl
    gc.collect()
    return tst_pred, test_ids, val_score

print('Training function ready with 5-seed ensemble')

Training function ready with 5-seed ensemble


In [7]:
test_outputs = []
validation_scores = {}

for hz in FORECAST_WINDOWS:
    tst_pred, test_ids, val_score = train_single_horizon(hz)
    test_outputs.append(pd.DataFrame({'id': test_ids, 'prediction': tst_pred}))
    validation_scores[hz] = val_score

print('All horizons complete')

Training Horizon 1
Using 137 features
Horizon 1 Score: 0.07715
Training Horizon 3
Using 137 features
Horizon 3 Score: 0.13599
Training Horizon 10
Using 137 features
Horizon 10 Score: 0.21665
Training Horizon 25
Using 137 features
Horizon 25 Score: 0.27490
All horizons complete


In [8]:
submission = pd.concat(test_outputs, ignore_index=True)
submission.to_csv('submission.csv', index=False)
print(f'Saved {len(submission):,} predictions')
print(submission.head(10))

Saved 1,447,107 predictions
                                      id  prediction
0  10BAVIDU__07YQ9WA4__DPPUO5X2__1__4175   -0.022701
1  10BAVIDU__07YQ9WA4__DPPUO5X2__1__4176   -0.031947
2  10BAVIDU__07YQ9WA4__DPPUO5X2__1__4177   -0.010733
3  10BAVIDU__07YQ9WA4__DPPUO5X2__1__4178    0.019040
4  10BAVIDU__07YQ9WA4__DPPUO5X2__1__4179   -0.018937
5  10BAVIDU__07YQ9WA4__DPPUO5X2__1__4180   -0.016553
6  10BAVIDU__07YQ9WA4__DPPUO5X2__1__4182   -0.011837
7  10BAVIDU__07YQ9WA4__DPPUO5X2__1__4183   -0.023956
8  10BAVIDU__07YQ9WA4__DPPUO5X2__1__4184   -0.004692
9  10BAVIDU__07YQ9WA4__DPPUO5X2__1__4185   -0.044104


In [9]:
print(f"{'Horizon':<10} {'Val Score':<12}")
print('-'*25)
for hz in sorted(validation_scores.keys()):
    print(f'{hz:<10} {validation_scores[hz]:<12.5f}')

avg_val = np.mean(list(validation_scores.values()))
print('-'*25)
print('SUBMISSION done')

Horizon    Val Score   
-------------------------
1          0.07715     
3          0.13599     
10         0.21665     
25         0.27490     
-------------------------
SUBMISSION done
